# PARTE 2: Limpieza, Exploración y Modificaciones
Para los ejercicios 1 y 2, debes asumir que estás trabajando sobre una tabla de prueba llamada tb_reservas_prueba.

### Se creó una copia de tb_reservas llamada tb_reservas_prueba para realizar pruebas sin modificar la tabla original.


In [0]:
%sql
CREATE OR REPLACE TABLE hotel_dataandino.proyecto_hotel_db.tb_reservas_prueba AS
SELECT 
    *
FROM hotel_dataandino.proyecto_hotel_db.tb_reservas;

# 1. (CREATE / INSERT) 
El equipo comercial quiere hacer un experimento rápido. Crea una tabla temporal (tb_reservas_prueba) y usa el comando INSERT con SELECT para insertar únicamente las reservas que corresponden a 'City Hotel' y que fueron canceladas (is_canceled = 1).

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tb_reservas_prueba AS
SELECT *
FROM hotel_dataandino.proyecto_hotel_db.tb_reservas
WHERE hotel = 'City Hotel'
AND is_canceled = 1;

In [0]:
%sql
SHOW VIEWS;

### Interpretación del resultado:
- Se creó la vista temporal tb_reservas_prueba con las reservas de City Hotel que fueron canceladas (is_canceled = 1), para trabajar con estos registros sin modificar la tabla original tb_reservas.

# 2. (UPDATE) 
Se detectó un error en los sistemas de captura de datos: los huéspedes que registraron babies en realidad no tenían bebés, sino que era un error del canal. Actualiza la tabla tb_reservas_prueba y cambia el valor de babies a 0 para todos

In [0]:
%sql
update hotel_dataandino.proyecto_hotel_db.tb_reservas_prueba
set babies = 0
WHERE babies = 1;

### Interpretación del resultado:
- Se corrigieron un total de 900 registros en la columna babies, asignando el valor 0 para evitar trabajar con información incorrecta en tb_reservas_prueba.

# 3. (CAST / CONCAT) 
Necesitamos tener la fecha de llegada en un formato legible. Selecciona las columnas de año, mes (que viene en texto, ej. 'July') y día, y concaténalas en una nueva columna llamada arrival_date_string con el formato "DD-Mes-YYYY" (Ej: "25-July-2015").

In [0]:
%sql
SELECT
    TRIM(
        CONCAT(
            CAST(arrival_date_day_of_month AS STRING), '-',
            arrival_date_month, '-',
            CAST(arrival_date_year AS STRING)
              )
          ) AS arrival_date_string  
FROM  hotel_dataandino.proyecto_hotel_db.tb_reservas
limit 20;

### Interpretación del resultado:
- Se unieron el día, mes y año para crear una nueva columna llamada arrival_date_string. Así se obtuvo la fecha en el formato solicitado sin modificar las columnas originales.

# 4. (LIKE) 
Queremos filtrar a un segmento muy específico del canal de distribución. Selecciona todas las reservas donde distribution_channel empiece con la letra 'T'.

In [0]:
%sql
SELECT
    *
FROM  hotel_dataandino.proyecto_hotel_db.tb_reservas
WHERE distribution_channel like 'T%';

### Interpretación del resultado:
- Se filtraron las reservas cuyo canal de distribución comienza con la letra T utilizando la función LIKE. Así se identificaron únicamente los registros que cumplen con esa condición.

# 5. (COUNT / SUM / AVG) 
Calcula a nivel global para todo el hotel: la cantidad total de reservas, la suma total de solicitudes especiales (total_of_special_requests) y el promedio del precio por noche (adr).

In [0]:
%sql
SELECT
    count(*) as cantidad_total_reservas,
    sum(total_of_special_requests) as total_solicitudes_especiales,
    round(avg(adr),2) as promedio_precio_por_noche
FROM  hotel_dataandino.proyecto_hotel_db.tb_reservas;

### Interpretación del resultado:
- Se contabilizaron 119,390 reservas y se registraron 68,215 solicitudes especiales. Además, se obtuvo un precio promedio de 101.83 por noche.

# 6. (DELETE)
En tu tabla de pruebas (tb_reservas_prueba), elimina todas las reservas que tuvieron un precio por noche (adr) igual a 0.0, ya que se consideran pruebas del sistema y no reservas reales.

In [0]:
%sql
DELETE FROM hotel_dataandino.proyecto_hotel_db.tb_reservas
WHERE adr= 0.0;

### Interpretación del resultado:
- Se eliminaron 1,959 registros que tenían un precio por noche (adr) igual a cero, ya que se consideraron pruebas del sistema y no reservas reales.  

# 7. (GROUP BY) 
El gerente quiere saber cuántas reservas entraron por cada canal de distribución. Obtén la cantidad de reservas agrupadas por distribution_channel.

In [0]:
%sql
SELECT
    distribution_channel,
    count(*) as cantidad_de_reservas
FROM  hotel_dataandino.proyecto_hotel_db.tb_reservas
GROUP by distribution_channel
ORDER BY cantidad_de_reservas DESC;

### Interpretación del resultado:
- Se identificó que el canal TA/TO concentra la mayor cantidad de reservas con 96,913 registros. Le sigue Direct con 13,839 y Corporate con 6,485. Los canales GDS y Undefined presentan una cantidad mucho menor de reservas.

# 8. (CASE WHEN / Tipos de Datos) 
Crea una consulta que clasifique las reservas en base al precio (adr): Si es menor a 50 es "Económico", si está entre 50 y 150 es "Estándar", y mayor a 150 es "Premium".

In [0]:
%sql
SELECT
    adr,
    CASE
        WHEN adr < 50 THEN 'Económico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
        WHEN adr > 150 THEN 'Premium'
    END AS Rango
FROM hotel_dataandino.proyecto_hotel_db.tb_reservas
ORDER BY adr DESC;

### Interpretación del resultado:
- Se clasificaron las reservas según su precio por noche en tres categorías: Económico, Estándar y Premium. De esta manera se identificó el rango de precio correspondiente a cada reserva.

# 9. (Funciones de Fecha - DATE_SUB / DATE_ADD) 
(Considera que existe una columna llamada arrival_date de tipo DATE para este ejercicio). Sabemos el lead_time (días de anticipación). Usa la función de Databricks para restar los días de lead_time a la arrival_date y así obtener la fecha exacta en la que el cliente hizo la reservación. Llámala booking_date.

In [0]:
%sql
WITH reservas AS (
        SELECT
            lead_time,
            TO_DATE(
                CONCAT(
                    CAST(arrival_date_day_of_month AS STRING), '-',
                    arrival_date_month, '-',
                    CAST(arrival_date_year AS STRING)
                ),
                'd-MMMM-yyyy'
            ) AS arrival_date
        FROM hotel_dataandino.proyecto_hotel_db.tb_reservas
                )
SELECT
    arrival_date,
    lead_time,
    DATE_SUB(arrival_date, CAST(lead_time AS INT)) AS booking_date
FROM reservas;

### Interpretación del resultado:
- Se calculó la fecha en que cada cliente realizó su reserva. Para ello se restaron los días de anticipación (lead_time) a la fecha de llegada (arrival_date). Así se obtuvo la fecha estimada en la que se hizo cada reservación.

# 10. (Funciones Matemáticas) 
Queremos calcular una "cuota de riesgo". Selecciona el
lead_time, divídelo entre 30 (para obtener meses aproximados) y usa la función
CEIL() (redondeo hacia arriba) para mostrar el resultado en una columna llamada
meses_anticipacion_techo.

In [0]:
%sql
SELECT
    lead_time,
    CEIL(lead_time / 30) AS meses_anticipacion_techo
FROM hotel_dataandino.proyecto_hotel_db.tb_reservas;


### Interpretación del resultado:
- Se dividió lead_time entre 30 para obtener los meses aproximados de anticipación. Luego, se utilizó la función CEIL() para redondear el resultado hacia arriba y obtener un número entero. Así se obtuvo la cantidad aproximada de meses de anticipación de cada cliente.